In [5]:
import os
import torch
import matplotlib.pyplot as plt
from collections import Counter
import warnings

# 忽略 PyG 在加载模型时的一些安全告警
warnings.filterwarnings('ignore')

def analyze_graph_dataset(base_dir):
    splits = ["train", "val", "test"]

    for split in splits:
        file_path = os.path.join(base_dir, split, "graphs.pt")
        if not os.path.exists(file_path):
            print(f"⚠️ 未找到文件: {file_path}")
            continue

        print(f"\n" + "="*50)
        print(f" 🔍 正在深度分析数据集: {split.upper()}")
        print(f"==================================================")

        try:
            # 开启 weights_only=False 允许加载 PyG 的 Data 对象
            graphs = torch.load(file_path, weights_only=False)
        except Exception as e:
            print(f"❌ 加载 {file_path} 失败: {e}")
            continue

        total_graphs = len(graphs)
        print(f"✅ 【图的总数】: {total_graphs}\n")

        if total_graphs == 0:
            continue

        node_counts = []
        edge_counts = []

        for g in graphs:
            # 获取每张图的节点数和边数
            node_counts.append(g.num_nodes)
            edge_counts.append(g.num_edges)

        # 使用 Counter 进行频率统计
        node_dist = Counter(node_counts)
        edge_dist = Counter(edge_counts)

        # ==========================================
        # 1. 打印详细文本分布数据
        # ==========================================
        print(f"--- 🟢 节点数量分布 ({split}) ---")
        for k in sorted(node_dist.keys()):
            print(f"  节点数为 {k:<4} 的图的数量: {node_dist[k]}")

        print(f"\n--- 🔴 边数量分布 ({split}) ---")
        for k in sorted(edge_dist.keys()):
            print(f"  边数为 {k:<4} 的图的数量: {edge_dist[k]}")

        # ==========================================
        # 2. 绘制并保存直方图分布图
        # ==========================================
        plt.figure(figsize=(14, 6))

        # 节点分布子图 (左侧)
        plt.subplot(1, 2, 1)
        nodes_k, nodes_v = zip(*sorted(node_dist.items()))
        plt.bar(nodes_k, nodes_v, color='skyblue', edgecolor='black', alpha=0.8)
        plt.title(f'Node Count Distribution ({split.upper()})', fontsize=14)
        plt.xlabel('Number of Nodes per Graph', fontsize=12)
        plt.ylabel('Number of Graphs', fontsize=12)
        plt.grid(axis='y', linestyle='--', alpha=0.7)

        # 边分布子图 (右侧)
        plt.subplot(1, 2, 2)
        edges_k, edges_v = zip(*sorted(edge_dist.items()))
        plt.bar(edges_k, edges_v, color='salmon', edgecolor='black', alpha=0.8)
        plt.title(f'Edge Count Distribution ({split.upper()})', fontsize=14)
        plt.xlabel('Number of Edges per Graph', fontsize=12)
        plt.ylabel('Number of Graphs', fontsize=12)
        plt.grid(axis='y', linestyle='--', alpha=0.7)

        plt.tight_layout()

        # 图片保存在终端执行的当前目录下
        plot_path = f'{split}_graph_distribution.png'
        plt.savefig(plot_path, dpi=300)
        plt.close()

        print(f"\n📊 [{split.upper()}] 分布直方图已生成并保存至: ./{plot_path}")

if __name__ == "__main__":
    # 配置你的真实数据存放根目录
    DATA_DIR = "/tmp/pycharm_project_908/processed_data/CIC-IoMT-2024-session-srcIP_srcPort_dstIP_dstPort_proto/processed_graphs"

    analyze_graph_dataset(DATA_DIR)


 🔍 正在深度分析数据集: TRAIN
✅ 【图的总数】: 253827

--- 🟢 节点数量分布 (train) ---
  节点数为 1    的图的数量: 240126
  节点数为 2    的图的数量: 4342
  节点数为 3    的图的数量: 874
  节点数为 4    的图的数量: 340
  节点数为 5    的图的数量: 334
  节点数为 6    的图的数量: 492
  节点数为 7    的图的数量: 699
  节点数为 8    的图的数量: 834
  节点数为 9    的图的数量: 984
  节点数为 10   的图的数量: 1053
  节点数为 11   的图的数量: 975
  节点数为 12   的图的数量: 824
  节点数为 13   的图的数量: 669
  节点数为 14   的图的数量: 464
  节点数为 15   的图的数量: 338
  节点数为 16   的图的数量: 199
  节点数为 17   的图的数量: 140
  节点数为 18   的图的数量: 66
  节点数为 19   的图的数量: 36
  节点数为 20   的图的数量: 24
  节点数为 21   的图的数量: 6
  节点数为 22   的图的数量: 4
  节点数为 23   的图的数量: 1
  节点数为 24   的图的数量: 2
  节点数为 25   的图的数量: 1

--- 🔴 边数量分布 (train) ---
  边数为 1    的图的数量: 240126
  边数为 3    的图的数量: 4342
  边数为 5    的图的数量: 724
  边数为 6    的图的数量: 150
  边数为 7    的图的数量: 316
  边数为 8    的图的数量: 9
  边数为 9    的图的数量: 327
  边数为 10   的图的数量: 11
  边数为 11   的图的数量: 467
  边数为 12   的图的数量: 12
  边数为 13   的图的数量: 658
  边数为 14   的图的数量: 26
  边数为 15   的图的数量: 756
  边数为 16   的图的数量: 32
  边数为 17   的图的数量: 891
  边数为 18   的图的数量

In [2]:
import os
import torch
from collections import Counter
import warnings

# 忽略加载模型时的安全告警
warnings.filterwarnings('ignore')

# =========================================================
# 1. 字典映射：将 ID 映射到具体的 Type
# =========================================================
label_id_map = {
    'benign': 0,
    'dos hulk': 1,
    'dos slowhttptest': 2,
    'dos slowloris': 3,
    'dos goldeneye': 4,
    'portscan': 5,
    'ddos': 6,
    'ftp-patator': 7,
    'ssh-patator': 8,
    'bot': 9,
    'web attack – brute force': 10,
    'web attack – xss': 11,
    'web attack – sql injection': 12,
    'infiltration': 13,
    'heartbleed': 14,
    'mixed': -1   # 混合攻击/未知类的兜底
}
id_to_type = {v: k for k, v in label_id_map.items()}

# =========================================================
# 2. 归属映射：将具体的 Type 归类到大类的 Family
# =========================================================
type_to_family = {
    'benign': 'Benign (正常)',
    'dos hulk': 'DoS',
    'dos slowhttptest': 'DoS',
    'dos slowloris': 'DoS',
    'dos goldeneye': 'DoS',
    'portscan': 'PortScan',
    'ddos': 'DDoS',
    'ftp-patator': 'BruteForce',
    'ssh-patator': 'BruteForce',
    'bot': 'Bot',
    'web attack – brute force': 'Web Attack',
    'web attack – xss': 'Web Attack',
    'web attack – sql injection': 'Web Attack',
    'infiltration': 'Infiltration',
    'heartbleed': 'Heartbleed',
    'mixed': 'Mixed/Ignored (混合)'
}

base_dir = "/tmp/pycharm_project_908/processed_data/CIC-IDS-2017-session-srcIP_srcPort_dstIP_dstPort_proto/processed_graphs"
splits = ["train", "val", "test"]

total_type_counts = Counter()
total_family_counts = Counter()
total_graphs = 0

print("="*60)
print(" 📊 CIC-IDS-2017 纯净图标签【全局总计】分布统计")
print("="*60)

# 统一读取所有数据并聚合
for split in splits:
    file_path = os.path.join(base_dir, split, "graphs.pt")
    if not os.path.exists(file_path):
        continue

    try:
        graphs = torch.load(file_path, weights_only=False)
        total_graphs += len(graphs)

        # 遍历每张图提取标签
        for g in graphs:
            y_m = g.y_multi.item() if hasattr(g, 'y_multi') else -1
            type_name = id_to_type.get(y_m, 'mixed')
            family_name = type_to_family.get(type_name, 'Unknown')

            total_type_counts[type_name] += 1
            total_family_counts[family_name] += 1

    except Exception as e:
        print(f"❌ 加载 {file_path} 失败: {e}")

if total_graphs == 0:
    print("⚠️ 未找到任何图数据，请检查路径。")
    exit()

print(f"\n 📂 聚合完毕 | 全局总会话图数: {total_graphs}")
print("-" * 60)

# ----- 打印家族分布 (宏观) -----
print("【1. 攻击家族 (OVR Family) 整体分布】:")
for family, count in sorted(total_family_counts.items(), key=lambda x: x[1], reverse=True):
    percentage = (count / total_graphs) * 100
    print(f"  🔸 {family:<20}: {count:>6} 张图 ({percentage:>5.1f}%)")

# ----- 打印类型分布 (微观) -----
print("\n【2. 攻击类型 (Fine-grained Type) 整体分布】:")
for t_name, count in sorted(total_type_counts.items(), key=lambda x: x[1], reverse=True):
    percentage = (count / total_graphs) * 100
    print(f"  🔹 {t_name:<20}: {count:>6} 张图 ({percentage:>5.1f}%)")

print("\n" + "="*60)

 📊 CIC-IDS-2017 纯净图标签【全局总计】分布统计

 📂 聚合完毕 | 全局总会话图数: 8402
------------------------------------------------------------
【1. 攻击家族 (OVR Family) 整体分布】:
  🔸 Benign (正常)         :   4232 张图 ( 50.4%)
  🔸 DoS                 :   1727 张图 ( 20.6%)
  🔸 PortScan            :   1561 张图 ( 18.6%)
  🔸 DDoS                :    787 张图 (  9.4%)
  🔸 BruteForce          :     63 张图 (  0.7%)
  🔸 Mixed/Ignored (混合)  :     12 张图 (  0.1%)
  🔸 Bot                 :     11 张图 (  0.1%)
  🔸 Web Attack          :      9 张图 (  0.1%)

【2. 攻击类型 (Fine-grained Type) 整体分布】:
  🔹 benign              :   4232 张图 ( 50.4%)
  🔹 dos hulk            :   1586 张图 ( 18.9%)
  🔹 portscan            :   1561 张图 ( 18.6%)
  🔹 ddos                :    787 张图 (  9.4%)
  🔹 dos goldeneye       :     72 张图 (  0.9%)
  🔹 ftp-patator         :     46 张图 (  0.5%)
  🔹 dos slowloris       :     45 张图 (  0.5%)
  🔹 dos slowhttptest    :     24 张图 (  0.3%)
  🔹 ssh-patator         :     17 张图 (  0.2%)
  🔹 mixed               :     12 张图 (  0.1%)
  🔹 bo

In [ ]:
import os
import pandas as pd
from collections import Counter

# =========================================================
# 1. 归属映射：将具体的 Type 归类到大类的 Family
# =========================================================
# 注意：这里我们使用小写来进行模糊或精确匹配，以兼容不同格式的原始标签
type_to_family = {
    'benign': 'Benign (正常)',
    'dos hulk': 'DoS',
    'dos slowhttptest': 'DoS',
    'dos slowloris': 'DoS',
    'dos goldeneye': 'DoS',
    'portscan': 'PortScan',
    'ddos': 'DDoS',
    'ftp-patator': 'BruteForce',
    'ssh-patator': 'BruteForce',
    'bot': 'Bot',
    'web attack – brute force': 'Web Attack',
    'web attack – xss': 'Web Attack',
    'web attack – sql injection': 'Web Attack',
    'infiltration': 'Infiltration',
    'heartbleed': 'Heartbleed'
}

def normalize_label(raw_label):
    if pd.isna(raw_label):
        return 'unknown'
    return str(raw_label).strip().lower()

def get_family_from_type(norm_type):
    # 尝试精确匹配
    if norm_type in type_to_family:
        return type_to_family[norm_type]

    # 尝试包含匹配 (处理一些奇怪的后缀或空格)
    for k, v in type_to_family.items():
        if k in norm_type:
            return v

    return 'Unknown/Other'

# 目标文件路径
csv_path = "/tmp/pycharm_project_908/processed_data/CIC-IDS-2017-session-srcIP_srcPort_dstIP_dstPort_proto/all_embedded_flow.filtered_port_53_67_68_123__svc_arp_dhcp_dns_llmnr_mdns_nbns_ntp.filtered_Heartbleed_Infiltration_Web_Attack___Sql_Injecti_6a9458d8.csv"

print("="*70)
print(" 📊 CIC-IDS-2017 过滤后底层【网络流 (Flow)】分布统计")
print("="*70)

if not os.path.exists(csv_path):
    print(f"⚠️ 未找到文件: {csv_path}")
    exit()

# =========================================================
# 2. 极速读取与统计 (防 OOM 策略)
# =========================================================
print("🔍 正在探测文件的标签列...")
# 先读 1 行获取列名
df_peek = pd.read_csv(csv_path, nrows=1)
label_col = next((col for col in ['label', 'Label', 'attack_type', 'Attack_Type'] if col in df_peek.columns), None)

if not label_col:
    print(f"❌ 无法在列名中找到标签列！当前列前20个为: {list(df_peek.columns)[:20]}")
    exit()

print(f"✅ 成功锁定标签列: [{label_col}]，开始极速读取...")
# 仅读取这一列，速度极快且不占内存
df_labels = pd.read_csv(csv_path, usecols=[label_col], low_memory=False)

total_flows = len(df_labels)
print(f"\n📂 读取完毕 | 过滤后的总网络流数量: {total_flows}")
print("-" * 70)

total_type_counts = Counter()
total_family_counts = Counter()

# 统计频次 (比遍历快很多)
raw_counts = df_labels[label_col].value_counts().to_dict()

for raw_label, count in raw_counts.items():
    norm_type = normalize_label(raw_label)
    family_name = get_family_from_type(norm_type)

    total_type_counts[norm_type] += count
    total_family_counts[family_name] += count

# =========================================================
# 3. 打印精美报表
# =========================================================
# ----- 打印家族分布 (宏观) -----
print("【1. 攻击家族 (OVR Family) 流级别分布】:")
for family, count in sorted(total_family_counts.items(), key=lambda x: x[1], reverse=True):
    percentage = (count / total_flows) * 100
    print(f"  🔸 {family:<20}: {count:>8} 条流 ({percentage:>6.2f}%)")

# ----- 打印类型分布 (微观) -----
print("\n【2. 攻击类型 (Fine-grained Type) 流级别分布】:")
for t_name, count in sorted(total_type_counts.items(), key=lambda x: x[1], reverse=True):
    percentage = (count / total_flows) * 100
    print(f"  🔹 {t_name:<20}: {count:>8} 条流 ({percentage:>6.2f}%)")

print("\n" + "="*70)

In [3]:
import os
import torch
import pandas as pd
from collections import Counter

# ==========================================
# 1. 路径配置 (请确保路径与你服务器一致)
# ==========================================
csv_path = "/tmp/pycharm_project_908/processed_data/CIC-IoMT-2024-session-srcIP_srcPort_dstIP_dstPort_proto/all_embedded_flow.filtered_port_53_67_68_123__svc_arp_dhcp_dns_llmnr_mdns_nbns_ntp.filtered_malicious_ARP_Spoofing_malicious_MQTT_DDoS_Conne_e1f622ff.merged_benign.csv"
graphs_base_dir = "/tmp/pycharm_project_908/processed_data/CIC-IoMT-2024-session-srcIP_srcPort_dstIP_dstPort_proto/processed_graphs"

# ==========================================
# 2. 映射逻辑
# ==========================================
# (A) 会话图 ID -> 家族 映射 (根据你训练代码的真实ID映射)
id_to_family = {
    0: "Benign",
    8: "Recon",
    9: "Recon",
    12: "DDoS",
    15: "DDoS",
    16: "DoS",
    19: "DoS",
    20: "DDoS",
    23: "DoS"
}

# (B) CSV 流标签名称 -> 家族 映射
def get_flow_family(label_str):
    l = str(label_str).lower()
    if 'ddos' in l: return 'DDoS'
    elif 'dos' in l: return 'DoS'
    elif 'recon' in l or 'scan' in l or 'sweep' in l: return 'Recon'
    elif 'benign' in l or 'normal' in l or 'legitimate' in l: return 'Benign'
    else: return 'Unknown'

print("🚀 开始全链路数据一致性核算...\n")

# ==========================================
# 3. 统计底层网络流 (Flows)
# ==========================================
print("⏳ 正在读取底层 CSV 文件统计流数量 (文件较大，请稍候)...")
flow_family_counts = Counter()
total_flows = 0

try:
    # 动态寻找标签列 (可能是 'label', 'Label', 'attack_type' 等)
    df_preview = pd.read_csv(csv_path, nrows=5)
    label_col = next((col for col in df_preview.columns if 'label' in col.lower() or 'attack' in col.lower()), None)

    if label_col:
        # 只读取标签列以节省内存
        df_labels = pd.read_csv(csv_path, usecols=[label_col])
        for label in df_labels[label_col]:
            fam = get_flow_family(label)
            flow_family_counts[fam] += 1
            total_flows += 1
        print("✅ 流数量统计完成！\n")
    else:
        print("❌ 错误: 在 CSV 中找不到标签列！")
except Exception as e:
    print(f"❌ 读取 CSV 失败: {e}\n")


# ==========================================
# 4. 统计会话图 (Session Graphs)
# ==========================================
print("⏳ 正在加载 .pt 文件统计图数量...")
graph_family_counts = Counter()
total_graphs = 0
splits = ["train", "val", "test"]

for split in splits:
    file_path = os.path.join(graphs_base_dir, split, "graphs.pt")
    if os.path.exists(file_path):
        try:
            graphs = torch.load(file_path, weights_only=False)
            total_graphs += len(graphs)
            for g in graphs:
                y_m = g.y_multi.item() if hasattr(g, 'y_multi') else -1
                family = id_to_family.get(y_m, "Unknown")
                graph_family_counts[family] += 1
        except Exception as e:
            print(f"❌ 加载 {split} 集图数据失败: {e}")
    else:
        print(f"⚠️ 找不到图文件: {file_path}")

print("✅ 图数量统计完成！\n")

# ==========================================
# 5. 打印完美对齐的 LaTeX 表格数据源
# ==========================================
print("="*80)
print("📊 CIC-IoMT-2024 最终核算数据看板 (用于填入 LaTeX 表格)")
print("="*80)
print(f"{'Attack Family':<15} | {'Flows (流数量)':<18} | {'Flows 占比':<12} | {'Session Graphs (图数量)':<25}")
print("-" * 80)

families_order = ["Benign", "DoS", "DDoS", "Recon"]
for fam in families_order:
    f_count = flow_family_counts.get(fam, 0)
    g_count = graph_family_counts.get(fam, 0)
    ratio = (f_count / total_flows * 100) if total_flows > 0 else 0
    print(f"{fam:<15} | {f_count:<18,} | {ratio:>8.2f}%   | {g_count:<25,}")

print("-" * 80)
print(f"{'Total':<15} | {total_flows:<18,} | {'100.00%':<12} | {total_graphs:<25,}")
print("="*80)

# 如果出现 Unknown，打印警告
if flow_family_counts.get("Unknown", 0) > 0 or graph_family_counts.get("Unknown", 0) > 0:
    print(f"⚠️ 警告: 存在未能识别的类别 -> Flows Unknown: {flow_family_counts.get('Unknown', 0)}, Graphs Unknown: {graph_family_counts.get('Unknown', 0)}")

🚀 开始全链路数据一致性核算...

⏳ 正在读取底层 CSV 文件统计流数量 (文件较大，请稍候)...
✅ 流数量统计完成！

⏳ 正在加载 .pt 文件统计图数量...
✅ 图数量统计完成！

📊 CIC-IoMT-2024 最终核算数据看板 (用于填入 LaTeX 表格)
Attack Family   | Flows (流数量)        | Flows 占比     | Session Graphs (图数量)     
--------------------------------------------------------------------------------
Benign          | 28,687             |     5.93%   | 24,600                   
DoS             | 203,417            |    42.04%   | 61,950                   
DDoS            | 216,585            |    44.77%   | 195,540                  
Recon           | 35,134             |     7.26%   | 82,080                   
--------------------------------------------------------------------------------
Total           | 483,823            | 100.00%      | 364,170                  


In [2]:
import os
import torch

# 你的处理后图数据路径
base_dir = "/tmp/pycharm_project_908/processed_data/CIC-IoMT-2024-session-srcIP_srcPort_dstIP_dstPort_proto/processed_graphs"
splits = ["train", "val", "test"]

total_graphs = 0
benign_graphs = 0
malicious_graphs = 0

print("🚀 开始极速统计图级别的二分类（善意/恶意）占比...\n")

for split in splits:
    file_path = os.path.join(base_dir, split, "graphs.pt")
    if not os.path.exists(file_path):
        continue

    try:
        graphs = torch.load(file_path, weights_only=False)
        total_graphs += len(graphs)

        # 统计二分类标签
        for g in graphs:
            # y_bin == 0.0 为正常， 1.0 为恶意
            if g.y_bin.item() == 0.0:
                benign_graphs += 1
            else:
                malicious_graphs += 1

    except Exception as e:
        print(f"❌ 加载 {split} 集失败: {e}")

print("="*50)
print("📊 CIC-IoMT-2024 图级别 (Session Graph) 恶意占比统计")
print("="*50)
if total_graphs > 0:
    benign_ratio = (benign_graphs / total_graphs) * 100
    malicious_ratio = (malicious_graphs / total_graphs) * 100
    print(f"✅ 总关系图数量 : {total_graphs}")
    print(f"🟢 正常图 (Benign)  : {benign_graphs} 张 ({benign_ratio:.2f}%)")
    print(f"🔴 恶意图 (Malicious): {malicious_graphs} 张 ({malicious_ratio:.2f}%)")
else:
    print("⚠️ 未找到任何图数据！")
print("="*50)

🚀 开始极速统计图级别的二分类（善意/恶意）占比...

📊 CIC-IoMT-2024 图级别 (Session Graph) 恶意占比统计
✅ 总关系图数量 : 364170
🟢 正常图 (Benign)  : 24600 张 (6.76%)
🔴 恶意图 (Malicious): 339570 张 (93.24%)


In [6]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from torch_geometric.nn import global_mean_pool
from collections import defaultdict
import random

# ==========================================
# 1. 配置参数
# ==========================================
DATA_PATH = "/tmp/pycharm_project_908/processed_data/CIC-IDS-2017-session-srcIP_srcPort_dstIP_dstPort_proto/processed_graphs/test/graphs.pt"
MODEL_CKPT = "/tmp/pycharm_project_908/processed_data/CIC-IDS-2017-session-srcIP_srcPort_dstIP_dstPort_proto/saved_models/graphmae/graphmae-best-epoch=04-train_loss=3.9691.ckpt"
SAMPLES_PER_CLASS = 500

# 类别映射与颜色配置 (严格对应您的要求)
CLASS_MAP = {
    0: "Benign",
    1: "DoS",
    2: "PortScan",
    3: "DDoS",
    4: "Bot",
    5: "Web Attack",
    6: "BruteForce"
}

COLORS = {
    "Benign": "blue",
    "DoS": "orange",
    "PortScan": "green",
    "DDoS": "red",
    "Bot": "purple",
    "Web Attack": "pink",
    "BruteForce": "brown"  # 补充 BruteForce 颜色
}

# ==========================================
# 2. 加载数据与提取指纹
# ==========================================
print("🚀 正在加载图数据和模型...")
graphs = torch.load(DATA_PATH, weights_only=False)

# 加载训练好的 GraphMAE 模型以提取编码器输出
from src.models.session_graphmae.graphmae_model import SessionGraphMAE
model = SessionGraphMAE.load_from_checkpoint(MODEL_CKPT)
model.eval()
model.cuda()

class_buckets = defaultdict(list)

print("🔍 提取流量指纹中...")
with torch.no_grad():
    for g in graphs:
        g = g.cuda()
        # 提取编码器特征 (z) 并进行全局平均池化得到图指纹
        z = g.x
        for conv in model.encoder:
            z = conv(z, g.edge_index)

        # 得到图级别的表征 (Fingerprint)
        batch_idx = torch.zeros(g.x.size(0), dtype=torch.long, device=g.x.device)
        fingerprint = global_mean_pool(z, batch_idx).cpu().numpy().flatten()

        # 获取类别标签 (使用 y_multi)
        label_id = int(g.y_multi.item())
        if label_id in CLASS_MAP:
            label_name = CLASS_MAP[label_id]
            if len(class_buckets[label_name]) < SAMPLES_PER_CLASS:
                class_buckets[label_name].append(fingerprint)

# ==========================================
# 3. PCA 降维与绘图
# ==========================================
all_points = []
all_labels = []

for label_name, points in class_buckets.items():
    all_points.extend(points)
    all_labels.extend([label_name] * len(points))

all_points = np.array(all_points)

print(f"📉 正在执行 PCA 降维 (总点数: {len(all_points)})...")
pca = PCA(n_components=2)
reduced_data = pca.fit_transform(all_points)

plt.figure(figsize=(10, 8))
for label_name in COLORS.keys():
    indices = [i for i, l in enumerate(all_labels) if l == label_name]
    if not indices: continue

    plt.scatter(
        reduced_data[indices, 0],
        reduced_data[indices, 1],
        c=COLORS[label_name],
        label=label_name,
        alpha=0.6,
        edgecolors='white',
        linewidths=0.5,
        s=40
    )

plt.title("MultiviewGraphMAE Traffic Fingerprint Visualization (PCA)", fontsize=14)
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.legend(loc='best')
plt.grid(True, linestyle='--', alpha=0.3)

save_path = "traffic_fingerprint_pca.png"
plt.savefig(save_path, dpi=300)
print(f"✅ 可视化结果已保存至: {save_path}")
plt.show()

🚀 正在加载图数据和模型...


ModuleNotFoundError: No module named 'src'